# Using snail to get information for Jamaica q1500 river network, dem and snapped points

## Step 1: import files 

In [ ]:
!which python

In [ ]:
import os
import pyproj
from pyproj import Geod
from pathlib import Path

In [ ]:
import snail.damages
import snail.intersection
import snail.io

In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd

In [ ]:
print(os.path.exists("/opt/miniconda3/envs/snail_env/share/proj"))

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")

# points paths
RP_1500_snapped_points_100_rivers_1000distance_threshold_path = base_path / "RP_1500_points_snapped_1000threshold.gpkg"
RP_1500_points_100_rivers_path = base_path / "JM_FLRF_UD_Q1500_POINTS_EPSG3448.gpkg"

# raster paths
dem_filled = base_path / "dem_filled.tif"
rp1500_flood_path = base_path / "JM_FLRF_UD_Q1500_RD_02.tif"

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

#### First time around, with snapped points - to find DEM grid index
###### hazard_files = pd.DataFrame({"path": ["path/to/dem.tif"]})

In [ ]:
dem_metadata = pd.DataFrame({
    "path": [dem_filled]
})

In [ ]:
# Extend raster metadata
dem_metadata, grids = snail.io.extend_rasters_metadata(dem_metadata)

In [ ]:
assert len(grids) == 1
grid = grids[0]

In [ ]:
snapped_points = gpd.read_file(RP_1500_snapped_points_100_rivers_1000distance_threshold_path)

In [ ]:
# ensure points are in the grid CRS before calculating indices
prepared = snail.intersection.prepare_points(snapped_points).to_crs(grid.crs)
prepared.crs

In [ ]:
# push into split_linestrings
with_indices = snail.intersection.apply_indices(
    prepared, grid, index_i="dem_i", index_j="dem_j"
)

In [ ]:
# overwrite with_indices.geometry with original snapped_points.geometry to get Jamaica CRS back again

orig_crs = snapped_points.crs

# # swap geometry to the original
# with_indices.set_geometry(snapped_points.geometry.values, inplace=True)

# # the coordinates are now in orig_crs; update the CRS metadata (no reprojection!)
# with_indices.set_crs(orig_crs, inplace=True, allow_override=True)



orig_crs = snapped_points.crs
with_indices = with_indices.to_crs(orig_crs)  # note the assignment

In [ ]:
with_indices.crs

In [ ]:
# save to file
output_snapped_points_with_dem = base_path / "snapped_points_with_dem_indices.parquet"
with_indices.to_parquet(output_snapped_points_with_dem)

#### Second time around, with pre-snapped points - to find flood raster grid index
###### hazard_files = pd.DataFrame({"path": ["path/to/rp1500.tif"]


In [ ]:
flood_metadata = pd.DataFrame({
    "path": [rp1500_flood_path]
})

In [ ]:
# Extend raster metadata
flood_metadata, grids = snail.io.extend_rasters_metadata(flood_metadata)

In [ ]:
assert len(grids) == 1
grid = grids[0]

In [ ]:
non_snapped_points = gpd.read_file(RP_1500_points_100_rivers_path)

In [ ]:
grid.crs

In [ ]:
non_snapped_points.crs

In [ ]:
# push into points, flag to disable
prepared_2 = snail.intersection.prepare_points(non_snapped_points).to_crs(grid.crs)

In [ ]:
prepared_2.crs

In [ ]:
# push into split_linestrings
with_indices_2 = snail.intersection.apply_indices(
    prepared_2, grid, index_i="flood_i", index_j="flood_j"
)

In [ ]:
# overwrite with_indices.geometry with original snapped_points.geometry to get Jamaica CRS back again


# orig_crs_2 = non_snapped_points.crs

# # swap geometry to the original
# with_indices_2.set_geometry(non_snapped_points.geometry.values, inplace=True)

# # the coordinates are now in orig_crs; update the CRS metadata (no reprojection!)
# with_indices_2.set_crs(orig_crs_2, inplace=True, allow_override=True)


orig_crs_2 = non_snapped_points.crs
with_indices_2 = with_indices_2.to_crs(orig_crs_2)  # note the assignment

In [ ]:
with_indices_2.crs

In [ ]:
# save to file
output_non_snapped_points_with_flood = base_path / "non_snapped_points_with_flood_indices.parquet"
with_indices_2.to_parquet(output_non_snapped_points_with_flood)